download from companies marketcap

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
import requests
import os
from datetime import date

# --- Configuração ---
URL = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
OUTPUT_DIR = "../data/raw/"
FILE_PREFIX = "top_companies_revenue"
# --------------------

today = date.today().strftime("%Y%m%d")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, f"{FILE_PREFIX}_{today}.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)

response = requests.get(URL, headers={"User-Agent": "Mozilla/5.0"})
response.raise_for_status()

with open(OUTPUT_PATH, "wb") as f:
    f.write(response.content)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Tamanho: {len(response.content):,} bytes")


ConnectionError: HTTPSConnectionPool(host='companiesmarketcap.com', port=443): Max retries exceeded with url: /largest-companies-by-revenue/?download=csv (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000029D344FA910>: Failed to resolve 'companiesmarketcap.com' ([Errno 11001] getaddrinfo failed)"))

create interim dataset with standarlized columns

In [2]:
import glob
files = sorted(glob.glob("../data/raw/top_companies_revenue_*.csv"))
raw_data_file = files[-1]

print(f"Arquivo mais recente encontrado: {raw_data_file}")

import pandas as pd
df = pd.read_csv(raw_data_file)

std_columns = {
    "Rank": "rank",
    "Name": "company_name",
    "Symbol": "stock_symbol",
    "revenue_ttm": "revenue_ttm",
    "price (USD)": "price_usd",
    "country": "company_country",
}
df = df.rename(columns=std_columns)

replacement_char = {
    ",": ". ",
    ";": ". ",
    ":": ". ",
    "_": "-",
    "[": "(",
    "]": ")",
    "{": "(",
    "}": ")",
    "\"": "",
    "%": " percent ",
    "$": " dollar ",
    "#": " number ",
    "@": " at ",
    "!": "",
    "*": "",
    "+": " plus ",
    "=": " equals ",
    "<": " less than ",
    ">": " greater than ",
    "/": " slash ",
    "\\": " backslash ",
    "|": " pipe ",
    "~": " tilde "
}

for char in replacement_char.keys():
    df["company_name"] = df["company_name"].str.replace(char, replacement_char[char], regex=False)
    df["stock_symbol"] = df["stock_symbol"].str.replace(char, replacement_char[char], regex=False)

df["company_name"] = df["company_name"].str.lower()
display(df)


Arquivo mais recente encontrado: ../data/raw\top_companies_revenue_20260610.csv


,rank,company_name,stock_symbol,revenue_ttm,price_usd,company_country
0,1,amazon,AMZN,742776000000,238.14000,United States
1,2,walmart,WMT,713163000000,119.87500,United States
2,3,apple,AAPL,451442000000,292.76000,United States
3,4,unitedhealth,UNH,449713000000,407.05000,United States
4,5,saudi aramco,2222.SR,448639900174,7.20927,Saudi Arabia
...,...,...,...,...,...,...
10855,10856,yellow cake plc,YCA.L,-456112000,7.18697,Jersey
10856,10857,pursuit attractions and hospitality,PRSU,-529856000,46.24500,United States
10857,10858,kinnevik,KINV-A.ST,-1359113330,7.56736,Sweden
10858,10859,secure energy services,SES.TO,-2151095178,15.63150,Canada


Get GICS classification

In [ ]:
import sys,json
sys.path.insert(0, "../src")

from gics_classifier import classify_gics

for company in df.head(5).to_dict(orient="records"):
    print(f"Classificando {company['company_name']}, {company['company_country']}, {company['stock_symbol']}...")
    result = classify_gics(company["company_name"], company["company_country"], company["stock_symbol"])
    print(json.dumps(result, indent=2))


Classificando amazon, United States, AMZN...
{
  "gics_sector": "Consumer Discretionary",
  "gics_industry_group": "Retailing",
  "gics_industry": "Internet and Direct Marketing Retail",
  "gics_source": "cache"
}
Classificando walmart, United States, WMT...
{
  "gics_sector": "Consumer Staples",
  "gics_industry_group": "Retailing",
  "gics_industry": "General Merchandise Stores",
  "gics_source": "cache"
}
Classificando apple, United States, AAPL...
{
  "gics_sector": "Information Technology",
  "gics_industry_group": "Technology Hardware & Equipment",
  "gics_industry": "Technology Hardware, Storage & Peripherals",
  "gics_source": "api"
}
Classificando unitedhealth, United States, UNH...
{
  "gics_sector": "Health Care",
  "gics_industry_group": "Health Care Providers & Services",
  "gics_industry": "Managed Health Care",
  "gics_source": "api"
}
Classificando saudi aramco, Saudi Arabia, 2222.SR...
{
  "gics_sector": "Energy",
  "gics_industry_group": "Energy",
  "gics_industry": "

: 